In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import LabelEncoder, StandardScaler, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tsa.stattools import adfuller

import shap

plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.dpi'] = 120
print("All libraries loaded successfully!")

In [ ]:
df = pd.read_csv(r'C:\Users\USER\OneDrive\Documents\Celebal Technologies\tesla_deliveries_dataset_2015_2025.csv')  # ← update path if needed

print("=" * 60)
print(f"Dataset Shape: {df.shape}")
print("=" * 60)
print(df.head(10))
print("\nColumn Dtypes:\n", df.dtypes)
print("\nBasic Statistics:")
df.describe().T

In [ ]:
print("─── Missing Values ───")
print(df.isnull().sum())

print("\n─── Unique Values per Categorical Column ───")
for col in ['Region', 'Model', 'Source_Type']:
    print(f"{col}: {df[col].unique()}")

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()

num_cols = ['Estimated_Deliveries', 'Production_Units', 'Avg_Price_USD',
            'Battery_Capacity_kWh', 'Range_km', 'CO2_Saved_tons']

for i, col in enumerate(num_cols):
    axes[i].hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(f'Distribution: {col}', fontsize=11)
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('Frequency')

plt.suptitle('Numerical Feature Distributions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:

# Yearly delivery trend
yearly = df.groupby('Year')['Estimated_Deliveries'].sum().reset_index()
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(yearly['Year'], yearly['Estimated_Deliveries'], marker='o', color='crimson', linewidth=2)
axes[0].set_title('Total Deliveries per Year')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Total Deliveries')

# Deliveries by Region
region_df = df.groupby('Region')['Estimated_Deliveries'].sum().sort_values(ascending=False)
axes[1].bar(region_df.index, region_df.values, color=['#2196F3','#FF5722','#4CAF50','#9C27B0'])
axes[1].set_title('Total Deliveries by Region')
axes[1].set_xlabel('Region'); axes[1].set_ylabel('Total Deliveries')

# Avg Price by Model
model_price = df.groupby('Model')['Avg_Price_USD'].mean().sort_values(ascending=False)
axes[2].barh(model_price.index, model_price.values, color='teal')
axes[2].set_title('Avg Price by Model (USD)')
axes[2].set_xlabel('Avg Price USD')

plt.suptitle('EDA: Temporal & Category Insights', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Correlation heatmap
plt.figure(figsize=(10, 7))
corr = df.select_dtypes(include=np.number).corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix of Numerical Features', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Price over time
df_sorted = df.groupby('Year')['Avg_Price_USD'].mean().reset_index()
axes[0].plot(df_sorted['Year'], df_sorted['Avg_Price_USD'], marker='s', color='darkorange', linewidth=2)
axes[0].set_title('Avg Price Trend (2015-2025)')
axes[0].set_xlabel('Year'); axes[0].set_ylabel('Avg Price USD')

# Model-wise delivery boxplot
model_order = df.groupby('Model')['Estimated_Deliveries'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='Model', y='Estimated_Deliveries', order=model_order,
            palette='Set2', ax=axes[1])
axes[1].set_title('Delivery Distribution by Model')
axes[1].set_xlabel('Model'); axes[1].set_ylabel('Estimated Deliveries')

plt.tight_layout()
plt.show()

# Scatter: Range vs Deliveries colored by model
plt.figure(figsize=(9, 6))
for model, grp in df.groupby('Model'):
    plt.scatter(grp['Range_km'], grp['Estimated_Deliveries'], label=model, alpha=0.5)
plt.xlabel('Range (km)')
plt.ylabel('Estimated Deliveries')
plt.title('Range vs Deliveries by Model')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
df_fe = df.copy()

# 1. Date features
df_fe['Quarter'] = ((df_fe['Month'] - 1) // 3) + 1
df_fe['Is_Q4'] = (df_fe['Quarter'] == 4).astype(int)
df_fe['Is_H2'] = (df_fe['Month'] >= 7).astype(int)
df_fe['Month_Sin'] = np.sin(2 * np.pi * df_fe['Month'] / 12)
df_fe['Month_Cos'] = np.cos(2 * np.pi * df_fe['Month'] / 12)

# 2. Business features
df_fe['Supply_Efficiency'] = df_fe['Estimated_Deliveries'] / (df_fe['Production_Units'] + 1)
df_fe['Price_Per_kWh'] = df_fe['Avg_Price_USD'] / df_fe['Battery_Capacity_kWh']
df_fe['Price_Per_km'] = df_fe['Avg_Price_USD'] / df_fe['Range_km']
df_fe['CO2_Per_Delivery'] = df_fe['CO2_Saved_tons'] / (df_fe['Estimated_Deliveries'] + 1)
df_fe['Infra_Utilization'] = df_fe['Estimated_Deliveries'] / (df_fe['Charging_Stations'] + 1)
df_fe['Years_Since_Start'] = df_fe['Year'] - 2015
df_fe['Log_Deliveries'] = np.log1p(df_fe['Estimated_Deliveries'])

# 3. Encode categoricals
le_region = LabelEncoder()
le_model = LabelEncoder()
le_source = LabelEncoder()
df_fe['Region_enc'] = le_region.fit_transform(df_fe['Region'])
df_fe['Model_enc'] = le_model.fit_transform(df_fe['Model'])
df_fe['Source_enc'] = le_source.fit_transform(df_fe['Source_Type'])

# 4. One-hot for Region and Model
df_fe = pd.get_dummies(df_fe, columns=['Region', 'Model', 'Source_Type'], drop_first=False)

print(f"Feature Engineering Done | Shape: {df_fe.shape}")
print("New features added:")
new_feats = ['Quarter','Is_Q4','Is_H2','Month_Sin','Month_Cos','Supply_Efficiency',
             'Price_Per_kWh','Price_Per_km','CO2_Per_Delivery','Infra_Utilization',
             'Years_Since_Start','Log_Deliveries']
print(new_feats)
df_fe.head()

In [ ]:

TARGET = 'Avg_Price_USD'   # PRIMARY TARGET: Price regression

# Features: drop all variants of the target, log-target, and raw text columns
DROP_COLS = [TARGET, 'Log_Deliveries', 'Region_enc', 'Model_enc', 'Source_enc']
FEATURE_COLS = [c for c in df_fe.columns if c not in DROP_COLS]

X = df_fe[FEATURE_COLS]
y = df_fe[TARGET]

# Ensure all boolean cols → int
X = X.astype({col: int for col in X.select_dtypes('bool').columns})

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f"Train size: {X_train.shape} | Test size: {X_test.shape}")
print(f"Target — Mean: {y.mean():.2f}, Std: {y.std():.2f}, Min: {y.min():.2f}, Max: {y.max():.2f}")

In [ ]:

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te, scaled=False):
    model.fit(X_tr, y_tr)
    y_pred = model.predict(X_te)
    rmse  = np.sqrt(mean_squared_error(y_te, y_pred))
    mae   = mean_absolute_error(y_te, y_pred)
    r2    = r2_score(y_te, y_pred)
    cv    = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2').mean()
    print(f"{'─'*55}")
    print(f"Model      : {name}")
    print(f"RMSE       : {rmse:>10.2f}")
    print(f"MAE        : {mae:>10.2f}")
    print(f"R² (Test)  : {r2:>10.4f}")
    print(f"R² (CV-5)  : {cv:>10.4f}")
    return {'Model': name, 'RMSE': rmse, 'MAE': mae, 'R2_Test': r2, 'R2_CV': cv,
            'y_pred': y_pred}

results = []
baselines = [
    ('Linear Regression',  LinearRegression(),              X_train_sc, X_test_sc, True),
    ('Ridge Regression',   Ridge(alpha=1.0),                X_train_sc, X_test_sc, True),
    ('Lasso Regression',   Lasso(alpha=10),                 X_train_sc, X_test_sc, True),
    ('ElasticNet',         ElasticNet(alpha=10, l1_ratio=0.5), X_train_sc, X_test_sc, True),
]

for name, model, Xtr, Xte, sc in baselines:
    res = evaluate_model(name, model, Xtr, y_train, Xte, y_test)
    results.append(res)

In [ ]:

tree_models = [
    ('Random Forest',    RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ('Gradient Boosting',GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42)),
    ('XGBoost',          XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                      subsample=0.8, colsample_bytree=0.8,
                                      random_state=42, verbosity=0)),
    ('LightGBM',         LGBMRegressor(n_estimators=300, learning_rate=0.05, num_leaves=63,
                                       random_state=42, verbose=-1)),
]

for name, model in tree_models:
    res = evaluate_model(name, model, X_train, y_train, X_test, y_test)
    results.append(res)

In [ ]:
# ─── FIX: Run this to define best_xgb and best_lgb both ───

from sklearn.model_selection import KFold, GridSearchCV
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

kf = KFold(n_splits=5, shuffle=True, random_state=42)

# ── XGBoost Tuning ──
print("⏳ Tuning XGBoost... (2-3 mins)")
param_grid_xgb = {
    'n_estimators':     [200, 300],
    'max_depth':        [4, 6],
    'learning_rate':    [0.05, 0.1],
    'subsample':        [0.8, 1.0],
    'colsample_bytree': [0.7, 0.9],
}
grid_xgb = GridSearchCV(
    XGBRegressor(random_state=42, verbosity=0, n_jobs=-1),
    param_grid_xgb, cv=kf, scoring='r2', n_jobs=-1, verbose=1
)
grid_xgb.fit(X_train, y_train)
best_xgb = grid_xgb.best_estimator_
print("✅ XGBoost done! Best params:", grid_xgb.best_params_)
print(f"   Best CV R²: {grid_xgb.best_score_:.4f}")

# ── LightGBM Tuning ──
print("\n⏳ Tuning LightGBM... (2-3 mins)")
param_grid_lgb = {
    'n_estimators':      [200, 400],
    'learning_rate':     [0.03, 0.05],
    'num_leaves':        [31, 63],
    'max_depth':         [-1, 6],
    'min_child_samples': [10, 20],
}
grid_lgb = GridSearchCV(
    LGBMRegressor(random_state=42, verbose=-1, n_jobs=-1),
    param_grid_lgb, cv=kf, scoring='r2', n_jobs=-1, verbose=1
)
grid_lgb.fit(X_train, y_train)
best_lgb = grid_lgb.best_estimator_
print("✅ LightGBM done! Best params:", grid_lgb.best_params_)
print(f"   Best CV R²: {grid_lgb.best_score_:.4f}")

In [ ]:

estimators = [
    ('xgb',  best_xgb),
    ('lgbm', best_lgb),
    ('rf',   RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)),
    ('gbm',  GradientBoostingRegressor(n_estimators=200, learning_rate=0.05, random_state=42)),
]
final_estimator = Ridge(alpha=1.0)

stacking = StackingRegressor(
    estimators=estimators,
    final_estimator=final_estimator,
    cv=5, n_jobs=-1
)
stacking.fit(X_train, y_train)

res_stack = evaluate_model('Stacking Ensemble', stacking, X_train, y_train, X_test, y_test)
results.append(res_stack)

In [ ]:

results_df = pd.DataFrame([{k: v for k, v in r.items() if k != 'y_pred'} for r in results])
results_df = results_df.sort_values('R2_Test', ascending=False).reset_index(drop=True)

print("\n" + "=" * 65)
print("MODEL COMPARISON — SORTED BY R² (TEST)")
print("=" * 65)
print(results_df.to_string(index=False))

# Bar plot of R² scores
plt.figure(figsize=(12, 5))
colors = ['gold' if i == 0 else 'steelblue' for i in range(len(results_df))]
bars = plt.bar(results_df['Model'], results_df['R2_Test'], color=colors, edgecolor='white')
plt.xticks(rotation=30, ha='right')
plt.ylabel('R² Score (Test)')
plt.title('Model Comparison — R² Score', fontsize=13, fontweight='bold')
for bar, val in zip(bars, results_df['R2_Test']):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9)
plt.ylim(0, 1.05)
plt.tight_layout()
plt.show()

In [ ]:
best_name = results_df.iloc[0]['Model']
best_result = next(r for r in results if r['Model'] == best_name)
y_pred_best = best_result['y_pred']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: Actual vs Predicted
axes[0].scatter(y_test, y_pred_best, alpha=0.4, color='royalblue', edgecolors='none')
mn, mx = y_test.min(), y_test.max()
axes[0].plot([mn, mx], [mn, mx], 'r--', linewidth=2, label='Perfect Fit')
axes[0].set_xlabel('Actual Avg_Price_USD')
axes[0].set_ylabel('Predicted Avg_Price_USD')
axes[0].set_title(f'{best_name}: Actual vs Predicted')
axes[0].legend()

# Residuals
residuals = y_test - y_pred_best
axes[1].scatter(y_pred_best, residuals, alpha=0.4, color='tomato', edgecolors='none')
axes[1].axhline(0, color='black', linewidth=1.5, linestyle='--')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title(f'{best_name}: Residual Plot')

plt.suptitle(f'Best Model: {best_name}  |  R²={best_result["R2_Test"]:.4f}  RMSE={best_result["RMSE"]:.2f}',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
explainer = shap.TreeExplainer(best_xgb)
shap_values = explainer.shap_values(X_test)

plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test, plot_type='bar',
                  feature_names=X_test.columns.tolist(),
                  show=False, max_display=15)
plt.title('SHAP Feature Importance (XGBoost Tuned)', fontsize=13)
plt.tight_layout()
plt.show()

# Beeswarm
plt.figure(figsize=(10, 7))
shap.summary_plot(shap_values, X_test,
                  feature_names=X_test.columns.tolist(),
                  show=False, max_display=15)
plt.title('SHAP Beeswarm — Feature Impact on Price Prediction', fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:

ts_df = df.copy()
ts_df['Date'] = pd.to_datetime(ts_df[['Year', 'Month']].assign(Day=1))
ts_monthly = (ts_df.groupby('Date')['Estimated_Deliveries']
                    .sum()
                    .sort_index()
                    .reset_index())
ts_monthly.columns = ['Date', 'Total_Deliveries']
ts_monthly = ts_monthly.set_index('Date')
ts_monthly = ts_monthly.asfreq('MS')  # Monthly Start frequency

plt.figure(figsize=(14, 4))
plt.plot(ts_monthly, color='darkcyan', linewidth=1.5)
plt.title('Monthly Tesla Global Deliveries (Aggregated)')
plt.xlabel('Date'); plt.ylabel('Total Deliveries')
plt.tight_layout()
plt.show()

# ADF Stationarity Test
adf_result = adfuller(ts_monthly['Total_Deliveries'].dropna())
print(f"ADF Statistic : {adf_result[0]:.4f}")
print(f"p-value       : {adf_result[1]:.4f}")
print("Stationary?" , "YES ✅" if adf_result[1] < 0.05 else "NO ❌ — consider differencing")

In [ ]:
split_date = ts_monthly.index[-13]
ts_train = ts_monthly[ts_monthly.index <= split_date]
ts_test  = ts_monthly[ts_monthly.index >  split_date]

hw_model = ExponentialSmoothing(
    ts_train['Total_Deliveries'],
    trend='add',
    seasonal='add',
    seasonal_periods=12,
    initialization_method='estimated'
).fit(optimized=True)

hw_pred = hw_model.forecast(len(ts_test))

hw_rmse = np.sqrt(mean_squared_error(ts_test['Total_Deliveries'], hw_pred))
hw_mae  = mean_absolute_error(ts_test['Total_Deliveries'], hw_pred)
hw_r2   = r2_score(ts_test['Total_Deliveries'], hw_pred)

print(f"Holt-Winters | RMSE={hw_rmse:.0f} | MAE={hw_mae:.0f} | R²={hw_r2:.4f}")

plt.figure(figsize=(14, 5))
plt.plot(ts_train.index, ts_train['Total_Deliveries'], label='Train', color='steelblue')
plt.plot(ts_test.index,  ts_test['Total_Deliveries'],  label='Actual (Test)', color='black', linewidth=2)
plt.plot(ts_test.index,  hw_pred.values,               label='Holt-Winters Forecast', color='crimson', linestyle='--', linewidth=2)
plt.title(f'Holt-Winters Forecast  |  R²={hw_r2:.4f}', fontsize=13)
plt.legend(); plt.tight_layout()
plt.show()

In [ ]:

sarima = SARIMAX(
    ts_train['Total_Deliveries'],
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 12),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarima_fit = sarima.fit(disp=False)
print(sarima_fit.summary())

sarima_pred = sarima_fit.get_forecast(steps=len(ts_test))
sarima_mean = sarima_pred.predicted_mean
sarima_ci   = sarima_pred.conf_int()

s_rmse = np.sqrt(mean_squared_error(ts_test['Total_Deliveries'], sarima_mean))
s_mae  = mean_absolute_error(ts_test['Total_Deliveries'], sarima_mean)
s_r2   = r2_score(ts_test['Total_Deliveries'], sarima_mean)
print(f"\nSARIMA | RMSE={s_rmse:.0f} | MAE={s_mae:.0f} | R²={s_r2:.4f}")

plt.figure(figsize=(14, 5))
plt.plot(ts_train.index, ts_train['Total_Deliveries'], label='Train', color='steelblue')
plt.plot(ts_test.index,  ts_test['Total_Deliveries'],  label='Actual (Test)', color='black', linewidth=2)
plt.plot(ts_test.index,  sarima_mean.values,           label='SARIMA Forecast', color='darkorange', linestyle='--', linewidth=2)
plt.fill_between(ts_test.index, sarima_ci.iloc[:, 0], sarima_ci.iloc[:, 1],
                 color='darkorange', alpha=0.15, label='95% CI')
plt.title(f'SARIMA(1,1,1)(1,1,1,12) Forecast  |  R²={s_r2:.4f}', fontsize=13)
plt.legend(); plt.tight_layout()
plt.show()

In [ ]:

# Refit on ALL data and forecast next 12 months
hw_full = ExponentialSmoothing(
    ts_monthly['Total_Deliveries'],
    trend='add', seasonal='add',
    seasonal_periods=12,
    initialization_method='estimated'
).fit(optimized=True)

future_steps = 12
future_idx = pd.date_range(ts_monthly.index[-1] + pd.DateOffset(months=1),
                            periods=future_steps, freq='MS')
future_pred = hw_full.forecast(future_steps)

plt.figure(figsize=(14, 5))
plt.plot(ts_monthly.index, ts_monthly['Total_Deliveries'],
         label='Historical', color='steelblue', linewidth=1.5)
plt.plot(future_idx, future_pred.values,
         label='Forecast (Next 12 Months)', color='crimson',
         linewidth=2.5, linestyle='--', marker='o')
plt.axvline(ts_monthly.index[-1], color='gray', linestyle=':', linewidth=1.5)
plt.title('Tesla Delivery Forecast — Next 12 Months', fontsize=13, fontweight='bold')
plt.xlabel('Date'); plt.ylabel('Total Deliveries')
plt.legend(); plt.tight_layout()
plt.show()

forecast_df = pd.DataFrame({'Date': future_idx, 'Forecasted_Deliveries': future_pred.values.round()})
print("\n─── 12-Month Forecast Table ───")
print(forecast_df.to_string(index=False))

In [ ]:

print("=" * 65)
print("            END-TO-END ML PIPELINE — FINAL SUMMARY")
print("=" * 65)

print("\n📊 REGRESSION RESULTS (Target: Avg_Price_USD)")
print(results_df[['Model','R2_Test','R2_CV','RMSE','MAE']].to_string(index=False))

print(f"\n🏆 BEST REGRESSION MODEL : {results_df.iloc[0]['Model']}")
print(f"   R² (Test)  : {results_df.iloc[0]['R2_Test']:.4f}")
print(f"   R² (CV-5)  : {results_df.iloc[0]['R2_CV']:.4f}")
print(f"   RMSE       : {results_df.iloc[0]['RMSE']:.2f}")
print(f"   MAE        : {results_df.iloc[0]['MAE']:.2f}")
